# Reproduction and Extension of a Metabolomics Study Using Unsupervised Machine Learning

#### Dataset: MTBLS79
#### Course: Machine Learning
#### Student: Sama Heidardoost

## Project Objective

The objective of this project is to reproduce the main analysis presented in the study "Direct infusion mass spectrometry metabolomics dataset: a benchmark for data processing and quality control" and extend the original work using additional unsupervised machine learning techniques.
The original study focused on data quality assessment and dimensionality reduction using PCA. In this project, additional methods such as UMAP and clustering algorithms are applied to further investigate the structure of the metabolomics dataset.

## Dataset Description

The MTBLS79 dataset is a metabolomics benchmark dataset generated using Direct Infusion Mass Spectrometry (DIMS).
The dataset contains metabolomic profiles obtained from cardiac tissue samples collected from cows and sheep. In addition, Quality Control (QC) samples were included to evaluate analytical reproducibility and data quality.
The processed dataset used in this project is Dataset10a__SFPM_PQN_KNN_GLOG.xlsx, which has already undergone normalization, missing value imputation and logarithmic transformation.

## Original Study Workflow

The original study applied the following preprocessing and analysis pipeline:
1. SFPM normalization
2. PQN normalization
3. KNN imputation
4. GLOG transformation
5. PCA analysis
The purpose of this project is first to reproduce the PCA analysis and then extend the study using additional unsupervised learning methods.

## Source Article
Kirwan, J. A., Weber, R. J. M., Broadhurst, D. I., & Viant, M. R. (2014).
Direct Infusion Mass Spectrometry Metabolomics Dataset: A Benchmark for Data Processing and Quality Control.
Scientific Data, 1, 140012.
DOI: 10.1038/sdata.2014.12

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import skfuzzy as fuzz
from minisom import MiniSom
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import Birch
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.cluster import MeanShift
from sklearn.cluster import OPTICS
from sklearn.cluster import SpectralClustering
from sklearn.cluster import AffinityPropagation
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import normalized_mutual_info_score
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import fcluster
from scipy.cluster.hierarchy import dendrogram
from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
import umap
import hdbscan
import warnings
warnings.filterwarnings('ignore')

E:\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
file_path="Dataset10a__SFPM_PQN_KNN_GLOG.xlsx"
xls=pd.ExcelFile(file_path)
print(xls.sheet_names)

FileNotFoundError: [Errno 2] No such file or directory: 'Dataset10a__SFPM_PQN_KNN_GLOG.xlsx'

In [ ]:
data=pd.read_excel(file_path,sheet_name="data")
meta=pd.read_excel(file_path,sheet_name="meta")
peak=pd.read_excel(file_path,sheet_name="peak")

In [ ]:
data.head()

In [ ]:
meta.head()

In [ ]:
peak.head()

## Exploratory Data Analysis (EDA)

### Dataset Overview

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isnull().sum().sum()

In [ ]:
meta["Class"].value_counts()

In [ ]:
meta.columns

## Feature Distribution

In [ ]:
x=data.iloc[:,1:]

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(x.iloc[:,0],bins=30,color='midnightblue')
plt.title("Distribution of First Feature")
plt.savefig('Distribution_of_First_Feature.png',dpi=300,bbox_inches='tight')
plt.show()

## Feature Standardization

In [ ]:
scaler=StandardScaler()
x_scaled=scaler.fit_transform(x)

In [ ]:
pd.DataFrame(x_scaled).describe()

## Reproduction of Principal Component Analysis (PCA)

In [ ]:
pca_original=PCA(n_components=2)
x_pca_original=pca_original.fit_transform(x)
print(pca_original.explained_variance_ratio_)
print(sum(pca_original.explained_variance_ratio_))

In [ ]:
pca_df=pd.DataFrame({"PC1": x_pca_original[:,0],"PC2": x_pca_original[:,1],"Class": meta["Class"]})
plt.figure(figsize=(10,7))
for cls in pca_df["Class"].unique():
    subset = pca_df[pca_df["Class"] == cls]
    plt.scatter(subset["PC1"],subset["PC2"],label=cls,alpha=0.7)
plt.xlabel(f"PC1 ({pca_original.explained_variance_ratio_[0]*100:.2f}%)")
plt.ylabel(f"PC2 ({pca_original.explained_variance_ratio_[1]*100:.2f}%)")
plt.title("PCA Projection of MTBLS79 Dataset")
plt.legend()
plt.savefig('PCA_Projection_of_MTBLS79_Dataset.png',dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
pca_full=PCA()
pca_full.fit(x)
cumulative_variance=np.cumsum(pca_full.explained_variance_ratio_)
plt.figure(figsize=(8,5))
plt.plot(cumulative_variance,marker='o',color='midnightblue')
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Explained Variance by PCA Components")
plt.grid()
plt.savefig('Explained_Variance_by_PCA_Components.png',dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
n_80=np.argmax(cumulative_variance >= 0.80) + 1
print(n_80)

## Extension of the Original Study Using UMAP

In [ ]:
reducer=umap.UMAP(n_neighbors=15,min_dist=0.1,random_state=42)
x_umap=reducer.fit_transform(x_scaled)
print(x_umap.shape)

In [ ]:
umap_df=pd.DataFrame({"UMAP1": x_umap[:,0],"UMAP2": x_umap[:,1],"Class": meta["Class"]})  

In [ ]:
plt.figure(figsize=(10,7))
for cls in umap_df["Class"].unique():
    subset=umap_df[umap_df["Class"] == cls]
    plt.scatter(subset["UMAP1"],subset["UMAP2"],label=cls,alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP Projection of MTBLS79 Dataset")
plt.legend()
plt.savefig("UMAP_Projection_of_MTBLS79_Dataset.png",dpi=300,bbox_inches="tight")
plt.show()

Clustering was performed on the standardized feature space. UMAP was used solely for visualization.

## PCA versus UMAP

UMAP produced a clearer separation of the biological groups than PCA, suggesting that nonlinear dimensionality reduction is more suitable for visualizing the complex structure of metabolomics data.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,6))
sns.scatterplot(x=x_pca_original[:,0],y=x_pca_original[:,1],hue=meta["Class"],ax=ax[0])
ax[0].set_title("PCA Projection")
sns.scatterplot(x=x_umap[:,0],y=x_umap[:,1],hue=meta["Class"],ax=ax[1])
ax[1].set_title("UMAP Projection")
plt.tight_layout()
plt.savefig("PCA_versus_UMAP.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
le=LabelEncoder()
y_true=le.fit_transform(meta["Class"])
sil_pca=silhouette_score(x_pca_original,y_true)
sil_umap=silhouette_score(x_umap,y_true)
print("PCA Silhouette =",sil_pca)
print("UMAP Silhouette =",sil_umap)

In [ ]:
comparison=pd.DataFrame({"Method":["PCA","UMAP"],"Silhouette":[sil_pca,sil_umap]})
plt.figure(figsize=(5,4))
sns.barplot(data=comparison,x="Method",y="Silhouette",color='midnightblue')
plt.title("PCA vs UMAP Separation Quality")
plt.savefig("PCA_vs_UMAP_Separation_Quality.png",dpi=300,bbox_inches="tight")
plt.show()

## K-Means Clustering

In [ ]:
inertia=[]
for k in range(1,11):
    km=KMeans(n_clusters=k,random_state=42,n_init=10)
    km.fit(x_scaled)
    inertia.append(km.inertia_)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1,11),inertia,marker='o',color='midnightblue')
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal k")
plt.grid(True)
plt.savefig("Elbow_Method_for_Optimal_k.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
kmeans=KMeans(n_clusters=3,random_state=42,n_init=10)
kmeans_labels=kmeans.fit_predict(x_scaled)
print(kmeans_labels[:10])

In [ ]:
umap_df["Cluster"]=kmeans_labels
umap_df.head()

In [ ]:
ct=pd.crosstab(meta["Class"],kmeans_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
sil_scaled=silhouette_score(x_scaled,kmeans_labels)
print("Silhouette Score =",sil_scaled)

In [ ]:
ari=adjusted_rand_score(meta["Class"],kmeans_labels)
nmi=normalized_mutual_info_score(meta["Class"],kmeans_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in sorted(umap_df["Cluster"].unique()):
    subset = umap_df[umap_df["Cluster"] == cluster]
    plt.scatter(subset["UMAP1"],subset["UMAP2"],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("K-Means Clusters on UMAP Projection")
plt.legend()
plt.savefig("KMeans_UMAP.png",dpi=300,bbox_inches="tight")
plt.show()

## Hierarchical Clustering

In [ ]:
linked=linkage(x_scaled,method='ward')
print(linked.shape)

In [ ]:
plt.figure(figsize=(15,6))
dendrogram(linked,truncate_mode='lastp',p=20,leaf_rotation=90,leaf_font_size=10)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Cluster")
plt.ylabel("Distance")
plt.tight_layout()
plt.savefig("Hierarchical_Dendrogram.png",dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
hier_labels=fcluster(linked,t=3,criterion='maxclust')
print("Unique labels:",np.unique(hier_labels))
print("Number of clusters:",len(np.unique(hier_labels)))

In [ ]:
ct=pd.crosstab(meta["Class"],hier_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
sil_scaled=silhouette_score(x_scaled,hier_labels)
print("Silhouette Score =", sil_scaled)

In [ ]:
ari=adjusted_rand_score(meta["Class"],hier_labels)
nmi=normalized_mutual_info_score(meta["Class"],hier_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(hier_labels):
    plt.scatter(x_umap[hier_labels == cluster,0],x_umap[hier_labels == cluster,1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Hierarchical Clustering on UMAP Projection")
plt.legend()
plt.savefig("Hierarchical_Clustering_on_UMAP_Projection.png",dpi=300,bbox_inches='tight')
plt.show()

## DBSCAN Clustering

In [ ]:
neighbors=NearestNeighbors(n_neighbors=5)
neighbors_fit=neighbors.fit(x_scaled)
distances, indices=neighbors_fit.kneighbors(x_scaled)
distances=np.sort(distances[:,4])
plt.figure(figsize=(8,5))
plt.plot(distances,color='midnightblue')
plt.xlabel("Samples")
plt.ylabel("5-NN Distance")
plt.title("K-distance Graph for DBSCAN")
plt.grid(True)
plt.savefig("DBSCAN_KDistance.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
dbscan=DBSCAN(eps=55,min_samples=5)
db_labels=dbscan.fit_predict(x_scaled)
print("Unique labels:",np.unique(db_labels))

In [ ]:
n_clusters=len(set(db_labels)) - (1 if -1 in db_labels else 0)
print("Number of clusters:",n_clusters)
noise_points=np.sum(db_labels == -1)
print("Noise points:",noise_points)

In [ ]:
ct=pd.crosstab(meta["Class"],db_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
mask=db_labels != -1
if len(np.unique(db_labels[mask])) > 1:
    sil_scaled = silhouette_score(x_scaled[mask],db_labels[mask])
    print("Silhouette Score (x_scaled) =",sil_scaled)
else:
    print("Silhouette Score cannot be calculated ""(only one cluster detected)")

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(db_labels):
    plt.scatter(x_umap[db_labels == cluster, 0],x_umap[db_labels == cluster, 1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("DBSCAN Clustering on UMAP Projection")
plt.legend()
plt.savefig("DBSCAN_Clustering_on_UMAP.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
print(pd.Series(db_labels).value_counts())

## Gaussian Mixture Model (GMM)

In [ ]:
gmm=GaussianMixture(n_components=3,random_state=42)
gmm_labels=gmm.fit_predict(x_scaled)
print(gmm_labels[:10])

In [ ]:
ct=pd.crosstab(meta["Class"],gmm_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
score=silhouette_score(x_scaled,gmm_labels)
print("Silhouette Score =",score)

In [ ]:
ari=adjusted_rand_score(meta["Class"],gmm_labels)
nmi=normalized_mutual_info_score(meta["Class"],gmm_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(gmm_labels):
    plt.scatter(x_umap[gmm_labels==cluster,0],x_umap[gmm_labels==cluster,1],label=f'Cluster {cluster}',alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("GMM Clustering on UMAP Projection")
plt.legend()
plt.savefig("GMM_Clustering_on_UMAP_Projection.png",dpi=300,bbox_inches="tight")
plt.show()

## Spectral Clustering

In [ ]:
spectral=SpectralClustering(n_clusters=3,affinity='nearest_neighbors',random_state=42)
spectral_labels=spectral.fit_predict(x_scaled)
print("Unique labels:",np.unique(spectral_labels))
print("Number of clusters:",len(np.unique(spectral_labels)))

In [ ]:
ct=pd.crosstab(meta["Class"],spectral_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
sil=silhouette_score(x_scaled,spectral_labels)
print("Silhouette Score =",sil)

In [ ]:
ari=adjusted_rand_score(meta["Class"],spectral_labels)
nmi=normalized_mutual_info_score(meta["Class"],spectral_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(spectral_labels):
    plt.scatter(x_umap[spectral_labels==cluster,0],x_umap[spectral_labels==cluster,1],label=f'Cluster {cluster}',alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Spectral Clustering on UMAP Projection")
plt.legend()
plt.savefig("Spectral_Clustering_on_UMAP_Projection.png",dpi=300,bbox_inches="tight")
plt.show()

## OPTICS Clustering

In [ ]:
optics=OPTICS(min_samples=5,xi=0.05,min_cluster_size=0.05)
optics_labels=optics.fit_predict(x_scaled)

In [ ]:
print("Unique labels:",np.unique(optics_labels))
print("Number of clusters:",len(np.unique(optics_labels)) - (1 if -1 in optics_labels else 0))
print("Noise points:",np.sum(optics_labels == -1))

In [ ]:
ct=pd.crosstab(meta["Class"],optics_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
mask=optics_labels != -1
if len(np.unique(optics_labels[mask])) > 1:
    sil=silhouette_score(x_scaled[mask],optics_labels[mask])
    print("Silhouette Score =", sil)
else:
    print("Less than 2 clusters found.")

In [ ]:
ari=adjusted_rand_score(meta["Class"],optics_labels)
nmi=normalized_mutual_info_score(meta["Class"],optics_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(optics_labels):
    plt.scatter(x_umap[optics_labels == cluster,0],x_umap[optics_labels == cluster,1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("OPTICS Clustering on UMAP Projection")
plt.legend()
plt.savefig("OPTICS_Clustering_on_UMAP.png",dpi=300,bbox_inches="tight")
plt.show()

## BIRCH Clustering


In [ ]:
birch=Birch(n_clusters=3,threshold=0.5)
birch_labels=birch.fit_predict(x_scaled)
print("Unique labels:",np.unique(birch_labels))
print("Number of clusters:",len(np.unique(birch_labels)))

In [ ]:
ct=pd.crosstab(meta["Class"],birch_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
sil=silhouette_score(x_scaled,birch_labels)
print("Silhouette Score =", sil)

In [ ]:
ari=adjusted_rand_score(meta["Class"],birch_labels)
nmi=normalized_mutual_info_score(meta["Class"],birch_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(birch_labels):
    plt.scatter(x_umap[birch_labels == cluster,0],x_umap[birch_labels == cluster,1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("BIRCH Clustering on UMAP Projection")
plt.legend()
plt.savefig("BIRCH_Clustering_on_UMAP_Projection.png",dpi=300,bbox_inches="tight")
plt.show()

## Affinity Propagation Clustering

In [ ]:
ap=AffinityPropagation(random_state=42)
ap_labels=ap.fit_predict(x_scaled)
print("Unique labels:",np.unique(ap_labels))
print("Number of clusters:",len(np.unique(ap_labels)))

In [ ]:
ct=pd.crosstab(meta["Class"],ap_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
sil=silhouette_score(x_scaled,ap_labels)
print("Silhouette Score =",sil)

In [ ]:
ari=adjusted_rand_score(meta["Class"],ap_labels)
nmi=normalized_mutual_info_score(meta["Class"],ap_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(ap_labels):
    plt.scatter(x_umap[ap_labels == cluster,0],x_umap[ap_labels == cluster,1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Affinity Propagation on UMAP Projection")
plt.legend(bbox_to_anchor=(1.05,1),loc="upper left")
plt.savefig("Affinity_Propagation_on_UMAP_Projection.png",dpi=300,bbox_inches="tight")
plt.show()

## HDBSCAN Clustering

In [ ]:
hdb=hdbscan.HDBSCAN(min_cluster_size=5,min_samples=5)
hdb_labels=hdb.fit_predict(x_scaled)
print("Unique labels:",np.unique(hdb_labels))

In [ ]:
n_clusters=len(set(hdb_labels))
if -1 in hdb_labels:
    n_clusters -= 1
print("Number of clusters:",n_clusters)
print("Noise points:",np.sum(hdb_labels == -1))

In [ ]:
ct=pd.crosstab(meta["Class"],hdb_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
mask=hdb_labels != -1
sil=silhouette_score(x_scaled[mask],hdb_labels[mask])
print("Silhouette Score =",sil)

In [ ]:
ari=adjusted_rand_score(meta["Class"],hdb_labels)
nmi=normalized_mutual_info_score(meta["Class"],hdb_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(hdb_labels):
    plt.scatter(x_umap[hdb_labels == cluster,0],x_umap[hdb_labels == cluster,1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("HDBSCAN Clustering on UMAP Projection")
plt.legend()
plt.savefig("HDBSCAN_Clustering_on_UMAP.png",dpi=300,bbox_inches="tight")
plt.show()

## Fuzzy C-Means Clustering

In [ ]:
cntr,u,u0,d,jm,p,fpc=fuzz.cluster.cmeans(x_scaled.T,c=3,m=2,error=0.005,maxiter=1000,init=None)
fcm_labels=np.argmax(u,axis=0)
print("Unique labels:", np.unique(fcm_labels))
print("Number of clusters:", len(np.unique(fcm_labels)))
print("Fuzzy Partition Coefficient (FPC):", fpc)

In [ ]:
ct=pd.crosstab(meta["Class"],fcm_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
sil=silhouette_score(x_scaled,fcm_labels)
print("Silhouette Score =",sil)

In [ ]:
ari=adjusted_rand_score(meta["Class"],fcm_labels)
nmi=normalized_mutual_info_score(meta["Class"],fcm_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(fcm_labels):
    plt.scatter(x_umap[fcm_labels == cluster, 0],x_umap[fcm_labels == cluster, 1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("Fuzzy C-Means Clustering on UMAP Projection")
plt.legend()
plt.savefig("Fuzzy_CMeans_Clustering_on_UMAP_Projection.png",dpi=300,bbox_inches="tight")
plt.show()

## Self-Organizing Map (SOM)

In [ ]:
som=MiniSom(x=3,y=1,input_len=x_scaled.shape[1],sigma=1.0,learning_rate=0.5,random_seed=42)
som.random_weights_init(x_scaled)
som.train_random(x_scaled, 1000)
som_labels=np.array([som.winner(x)[0]for x in x_scaled])
print("Unique labels:", np.unique(som_labels))
print("Number of clusters:", len(np.unique(som_labels)))

In [ ]:
ct=pd.crosstab(meta["Class"],som_labels,margins=True)
sns.heatmap(ct,annot=True,cmap="Blues",fmt="d")
display(ct)

In [ ]:
sil=silhouette_score(x_scaled, som_labels)
print("Silhouette Score =",sil)

In [ ]:
ari=adjusted_rand_score(meta["Class"],som_labels)
nmi=normalized_mutual_info_score(meta["Class"],som_labels)
print("ARI =",ari)
print("NMI =",nmi)

In [ ]:
plt.figure(figsize=(10,7))
for cluster in np.unique(som_labels):
    plt.scatter(x_umap[som_labels == cluster,0],x_umap[som_labels == cluster,1],label=f"Cluster {cluster}",alpha=0.7)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("SOM Clustering on UMAP Projection")
plt.legend()
plt.savefig("SOM_Clustering_on_UMAP_Projection.png",dpi=300,bbox_inches="tight")
plt.show()

## Results and Discussion

### Comparative Evaluation of Clustering Methods

In [ ]:
results=pd.DataFrame({"Method": ["K-Means","GMM","SOM","FCM","Spectral","HDBSCAN","OPTICS","Hierarchical","BIRCH","Affinity Propagation"],
                      "Silhouette": [0.1936,0.1936,0.1936,0.1634,0.1016,0.2160,0.2589,0.1618,0.1618,0.2140],
                      "ARI": [1.0,1.0,1.0,0.8096,0.7025,0.5334,0.5291,0.4380,0.4380,0.1723],
                      "NMI": [1.0,1.0,1.0,0.8265,0.7616,0.6712,0.6456,0.4926,0.4926,0.5177]})
display(results)

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
ax.axis('off')
table=ax.table(cellText=results.values,colLabels=results.columns,cellLoc='center',loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2,1.5)
plt.savefig("Clustering_Methods_Comparison.png",dpi=300,bbox_inches="tight")
plt.show()

#### Ranking of Clustering Algorithms

In [ ]:
results.sort_values(by=["ARI","NMI","Silhouette"],ascending=False)

### Performance Comparison of Clustering Algorithms

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(results.set_index("Method"),annot=True,cmap="YlGnBu")
plt.title("Comparison of Clustering Methods")
plt.tight_layout()
plt.savefig("Comparison_of_Clustering_Methods.png",dpi=300,bbox_inches="tight")
plt.show()

The heatmap highlights the superiority of K-Means, GMM, and SOM, which achieved the highest ARI and NMI scores.

### Comparative Evaluation of Clustering Algorithms

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=results,x="Method",y="ARI",color='midnightblue')
plt.xticks(rotation=45)
plt.title("Comparison of ARI Scores")
plt.tight_layout()
plt.savefig("Comparison_of_ARI_Scores.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=results,x="Method",y="NMI",color='midnightblue')
plt.xticks(rotation=45)
plt.title("Comparison of NMI Scores")
plt.tight_layout()
plt.savefig("Comparison_of_NMI_Scores.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=results,x="Method",y="Silhouette",color='midnightblue')
plt.xticks(rotation=45)
plt.title("Comparison of Silhouette Scores")
plt.tight_layout()
plt.savefig("Comparison_of_Silhouette_Scores.png",dpi=300,bbox_inches="tight")
plt.show()

## Final Conclusion

This study evaluated the clustering structure of the MTBLS79 metabolomics dataset using multiple unsupervised learning approaches. PCA and UMAP were first applied for dimensionality reduction and visualization, revealing a clear underlying biological structure within the data. UMAP provided a more distinct separation of the biological groups compared to PCA, highlighting its effectiveness for nonlinear metabolomics data visualization.

A total of eleven clustering algorithms were evaluated using Silhouette Score, Adjusted Rand Index (ARI), and Normalized Mutual Information (NMI). Among all methods, K-Means, Gaussian Mixture Models (GMM), and Self-Organizing Maps (SOM) achieved the best performance, obtaining perfect ARI and NMI scores of 1.0 and successfully recovering the three biological classes (C, QC, and S).

Although some density-based methods, such as OPTICS and HDBSCAN, produced higher Silhouette Scores, their agreement with the true biological labels was lower. This demonstrates the importance of combining internal and external validation metrics when assessing clustering quality.

Overall, the results indicate that the MTBLS79 dataset contains a strong and well-defined biological signal. K-Means, GMM, and SOM provided the most accurate and biologically meaningful clustering solutions, while UMAP proved to be a valuable extension for data visualization and interpretation.
